# TTS Retro — Old Macintosh Style

Exploring formant synthesizers to replicate the classic MacinTalk (1984) sound.

**Candidates:**
1. `espeak-ng` — formant synth, same family as MacinTalk, ARM64 native
2. `sam` — direct Python port of Software Automatic Mouth (the actual Mac/Atari engine)
3. `pyttsx3` — cross-platform wrapper, fallback

**Install before running:**
```bash
# Windows
winget install espeak-ng        # or: choco install espeak
pip install pyaudio numpy scipy pyttsx3

# Jetson / Ubuntu
sudo apt install espeak-ng ffmpeg -y
pip install pyaudio numpy scipy pyttsx3
```

In [2]:
import subprocess
import shutil
import sys

# Test text — classic Mac 1984 demo vibe
TEST_FR = "Bonjour, je suis votre guide au musée du Louvre."
TEST_EN = "Hello, I am your museum guide. Welcome to the Louvre."

def check_deps():
    print("espeak-ng:", "✓" if shutil.which("espeak-ng") else "✗ not found")
    print("ffmpeg:    ", "✓" if shutil.which("ffmpeg") else "✗ not found (optional, for conversion)")
    for pkg in ["numpy", "scipy", "pyaudio", "pyttsx3", "IPython"]:
        try:
            __import__(pkg)
            print(f"{pkg}:   ✓")
        except ImportError:
            print(f"{pkg}:   ✗ pip install {pkg}")

check_deps()

espeak-ng: ✗ not found
ffmpeg:     ✓
numpy:   ✓
scipy:   ✓
pyaudio:   ✓
pyttsx3:   ✓
IPython:   ✓


## 1. espeak-ng — formant synthesizer

MacinTalk parameters to emulate:
- Rate: ~120–150 wpm (original Mac was slow)
- Pitch: low-mid (~40)
- Amplitude: 80
- Voice: `en` or `fr` with `+croak` or `+whisper` variant
- Sample rate: 8000 Hz (original Mac audio chip)

espeak-ng outputs WAV → we downsample to 8kHz to get the classic lo-fi quality.

In [ ]:
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
from IPython.display import Audio, display
import tempfile, os

def espeak_synth(text, lang="fr", rate=120, pitch=40, amplitude=80,
                 target_sr=8000, variant=""):
    """
    Synthesize text with espeak-ng and optionally downsample to lo-fi quality.
    variant: '' | '+croak' | '+whisper' | '+f1' etc.
    """
    voice = f"{lang}{variant}"
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        out_path = f.name

    cmd = [
        "espeak-ng",
        "-v", voice,
        "-s", str(rate),        # words per minute
        "-p", str(pitch),       # pitch 0-99
        "-a", str(amplitude),   # amplitude 0-200
        "-w", out_path,         # write to WAV
        text
    ]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0:
        print("Error:", result.stderr.decode())
        return None, None

    sr, data = wavfile.read(out_path)
    os.unlink(out_path)

    # Downsample to target_sr for lo-fi Mac feel
    if target_sr and target_sr != sr:
        from math import gcd
        g = gcd(target_sr, sr)
        data = resample_poly(data.astype(np.float32), target_sr // g, sr // g)
        sr = target_sr

    return sr, data.astype(np.int16)


print("=== Preset 1: Classic Mac (French, slow, low pitch, 8kHz) ===")
sr, audio = espeak_synth(TEST_FR, lang="fr", rate=120, pitch=35, amplitude=80, target_sr=8000)
if audio is not None:
    display(Audio(audio, rate=sr))

In [ ]:
print("=== Preset 2: Full quality (22kHz) for comparison ===")
sr, audio = espeak_synth(TEST_FR, lang="fr", rate=120, pitch=35, target_sr=None)
if audio is not None:
    display(Audio(audio, rate=sr))

print("\n=== Preset 3: English, croak variant ===")
sr, audio = espeak_synth(TEST_EN, lang="en", rate=130, pitch=40, variant="+croak", target_sr=8000)
if audio is not None:
    display(Audio(audio, rate=sr))

print("\n=== Preset 4: Very slow, very low pitch (1984 Mac demo style) ===")
sr, audio = espeak_synth(TEST_EN, lang="en", rate=100, pitch=20, amplitude=90, target_sr=8000)
if audio is not None:
    display(Audio(audio, rate=sr))

In [ ]:
# Sweep pitch and rate to find the best Mac-like preset
print("=== Parameter sweep ===")
for pitch in [20, 35, 50]:
    for rate in [100, 130]:
        print(f"  pitch={pitch}, rate={rate}:")
        sr, audio = espeak_synth("Bienvenue au Louvre.", lang="fr",
                                  rate=rate, pitch=pitch, target_sr=8000)
        if audio is not None:
            display(Audio(audio, rate=sr))

## 2. SAM — Software Automatic Mouth

The actual engine from 1982 (Commodore 64, early Mac). Python port generates audio from phonemes directly.

```bash
pip install sam-tts
```

SAM works in phoneme space, so French text needs transliteration.
English works directly (it has a built-in text-to-phoneme converter).

In [ ]:
try:
    import sam
    HAS_SAM = True
except ImportError:
    HAS_SAM = False
    print("sam not installed — run: pip install sam-tts")

if HAS_SAM:
    from IPython.display import Audio, display
    import numpy as np

    # SAM presets — closer to original Mac
    configs = [
        {"name": "Default SAM",      "speed": 72, "pitch": 64, "throat": 128, "mouth": 128},
        {"name": "Classic Mac Fred", "speed": 60, "pitch": 60, "throat": 110, "mouth": 160},
        {"name": "Deep/Slow",        "speed": 50, "pitch": 40, "throat": 100, "mouth": 200},
    ]

    for cfg in configs:
        print(f"=== {cfg['name']} ===")
        audio_bytes = sam.render(
            TEST_EN,
            speed=cfg["speed"],
            pitch=cfg["pitch"],
            throat=cfg["throat"],
            mouth=cfg["mouth"],
        )
        # SAM outputs 8-bit unsigned, 22050 Hz
        audio = np.frombuffer(audio_bytes, dtype=np.uint8).astype(np.float32)
        audio = (audio - 128) / 128.0  # center around 0
        display(Audio(audio, rate=22050))

## 3. pyttsx3 — cross-platform fallback

Uses SAPI on Windows (modern sound), espeak on Linux. Good for quick tests but won't sound retro on Windows.

In [ ]:
try:
    import pyttsx3
    engine = pyttsx3.init()

    # List available voices
    voices = engine.getProperty('voices')
    print(f"{len(voices)} voices available:")
    for i, v in enumerate(voices):
        print(f"  [{i}] {v.id} — {v.name} ({v.languages})")

    # Tune for retro feel (low rate, lower pitch)
    engine.setProperty('rate', 130)   # default ~200
    engine.setProperty('volume', 0.9)

    # On Linux/Jetson: pick espeak voice
    # engine.setProperty('voice', voices[0].id)

    engine.say(TEST_FR)
    engine.runAndWait()
    print("✓ played via pyttsx3 (check your speakers)")

except ImportError:
    print("pyttsx3 not installed — pip install pyttsx3")
except Exception as e:
    print(f"pyttsx3 error: {e}")

## 4. Best preset — integration helper

Once you've picked a preset above, use this function to integrate with the Flask API.

In [ ]:
def speak_retro(text: str, lang: str = "fr") -> bytes:
    """
    Returns raw WAV bytes using the best retro preset found during experimentation.
    Drop this into src/tts.py once the preset is validated.
    """
    # === EDIT THESE after experimentation ===
    RATE      = 120   # words per minute
    PITCH     = 35    # 0-99
    AMPLITUDE = 80    # 0-200
    TARGET_SR = 8000  # Hz — set None for full quality
    # ========================================

    voice = lang
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        out_path = f.name

    subprocess.run([
        "espeak-ng",
        "-v", voice, "-s", str(RATE), "-p", str(PITCH),
        "-a", str(AMPLITUDE), "-w", out_path, text
    ], check=True)

    if TARGET_SR:
        sr, data = wavfile.read(out_path)
        from math import gcd
        g = gcd(TARGET_SR, sr)
        data = resample_poly(data.astype(np.float32), TARGET_SR // g, sr // g)
        wavfile.write(out_path, TARGET_SR, data.astype(np.int16))

    with open(out_path, "rb") as f:
        wav_bytes = f.read()
    os.unlink(out_path)
    return wav_bytes


# Test
wav = speak_retro(TEST_FR, lang="fr")
sr, data = wavfile.read(__import__("io").BytesIO(wav))
display(Audio(data, rate=sr))
print(f"WAV size: {len(wav)} bytes")

## Notes

| Engine | Sound | ARM64 | Install | French |
|--------|-------|-------|---------|--------|
| espeak-ng | Retro formant ★★★ | ✓ apt | Easy | ✓ |
| SAM | Authentic Mac 1984 ★★★★ | ✓ pip | Easy | Phonemes only |
| pyttsx3 | Modern (Windows SAPI) | ✓ | Easy | ✓ |

**Recommended for Jetson:** espeak-ng — installed via `apt`, zero Python dependency, minimal RAM, bilingual FR/EN.

**Most authentic Mac sound:** SAM — but English only without phoneme transliteration for French.

Next step: once preset validated, create `src/tts.py` and add a `GET /speak?text=...` endpoint to the Flask API.